In [0]:
%run "Workspace/Users/akhil.b@diggibyte.com/databricks_assignment/question_1/source_to_bronze/utils"

In [0]:
from pyspark.sql.types import *
from datetime import date
from pyspark.sql.functions import current_date, col, sum, desc , count , avg
import re

# Step 1: Read WITHOUT schema (to get correct order)
df = spark.read.option("header", True) \
    .csv("/Volumes/workspace/default/assignment/employee/")

# Step 2: Fix column order manually
df = df.select(
    "EmployeeID",
    "EmployeeName",
    "Department",
    "Salary",
    "Country",
    "Age"
)

# Step 3: Apply correct data types
employee_df = df.withColumn("EmployeeID", col("EmployeeID").cast("int")) \
    .withColumn("Salary", col("Salary").cast("int")) \
    .withColumn("Age", col("Age").cast("int"))

# Step 4: Convert to snake_case (if needed)
employee_df = employee_df.toDF(*[
    re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower()
    for c in employee_df.columns
])

# Step 5: Add load_date
employee_df = employee_df.withColumn("load_date", current_date())

# Step 6: Create database
spark.sql("CREATE DATABASE IF NOT EXISTS employee_info")

# Step 7: Write as Delta Table
employee_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("employee_info.dim_employee")


salary_by_dept = employee_df.groupBy("department") \
    .agg(sum("salary").alias("total_salary")) \
    .orderBy(col("total_salary").desc())



emp_count = employee_df.groupBy("department", "country") \
    .agg(count("employee_i_d").alias("employee_count"))

display(emp_count)


dept_country = employee_df.select("department", "country").distinct()

display(dept_country)


avg_age = employee_df.groupBy("department") \
    .agg(avg("age").alias("avg_age"))

display(avg_age)


employee_df = employee_df.withColumn("at_load_date", current_date())

display(employee_df)



gold_path = "/Volumes/workspace/default/assignment/gold/fact_employee"

today = date.today()

employee_df.write.format("delta").mode("overwrite").option("replaceWhere", f"at_load_date ={today}'").save(gold_path)

